In [4]:
import sys
import os

from datetime import datetime, timedelta, UTC
from dateutil import parser
from datetime import datetime, timedelta
import random

from faker import Faker
fake = Faker()

sys.path.append(os.path.abspath(".."))
# print(sys.path)

from mongo_get_database import get_database

db = get_database('agro_market', 'root', 'example')

### Tests

In [59]:
# products = {
#     _id: ObjectId,
#     name: '',
#     category: '',
#     description: '',
#     specs: {
#         moisture: 14.2,
#         protein 12.5,
#         gluten: 10.0
#     }
# }
# sellers = {
#     _id: ObjectId,
#     name: '',
#     country: '',
#     rating: 5.0
# }
# orders = {
#     _id: ObjectId,
#     product_id: ObjectId,
#     seller_id: ObjectId,
#     quantity: 10,
#     price: 100,
#     created_at: ISODate()
# }
# price_history = {
#     _id: ObjectId,
#     product_id: ObjectId,
#     price: 215,
#     date: ISODate()
# }

In [60]:
fake.pyfloat(2, 2, min_value=0, max_value=20)

16.97

### Drop

In [2]:
db['orders'].drop()
db['sellers'].drop()
db['products'].drop()
db['price_history'].drop()

### Inserts

In [3]:
def random_date(min_days=1, max_days=365):
    now = datetime.now()

    random_days = random.randint(1, 365)
    random_hours = random.randint(0, 23)
    random_minutes = random.randint(0, 59)

    random_date = now - timedelta(
        days=random_days,
        hours=random_hours,
        minutes=random_minutes
    )

    return parser.parse(random_date.isoformat())

In [4]:
products = db['products']
categories = ['corn', 'rapeseed', 'oats', 'millet', 'barley',
             'beans', 'chickpeas', 'soy', 'wheat']
products_data = []

for _ in range(200):
    products_data.append({
        'name': fake.word(),
        'categories': fake.random_element(categories),
        'description': fake.text(),
        'specs': {
            'moisture': fake.pyfloat(2, 2, min_value=0, max_value=20),
            'protein': fake.pyfloat(2, 2, min_value=0, max_value=80),
            'gluten': fake.pyfloat(2, 2, min_value=0, max_value=30)
        }
    })

result = products.insert_many(products_data)

In [5]:
sellers = db['sellers']
seller_data = []

for _ in range(200):
    seller_data.append({
        'name': fake.company(),
        'country': fake.country(),
        'rating': fake.pyfloat(2, 2, min_value=0, max_value=5)
    })

result = sellers.insert_many(seller_data)

In [6]:
products_id = [product['_id'] for product in products.find()]
sellers_id = [seller['_id'] for seller in sellers.find()]
orders = db['orders']
orders_data = []

for _ in range(1001):
    orders_data.append({
        'product_id': fake.random_element(products_id),
        'seller_id': fake.random_element(sellers_id),
        'quantity': fake.random_int(1, 950),
        'price': fake.random_int(100, 650),
        'created_at': random_date()
    })

result = orders.insert_many(orders_data)

In [7]:
price_history = db['price_history']
data = []

for pid in products_id:

    for _ in range(random.randint(1, 35)):
        data.append({
            'product_id': pid,
            'price': random.randint(50,900),
            'date': random_date(max_days=365)
        })

result = price_history.insert_many(data)

### Indexes

In [8]:
# [item for item in products.find()]
exp = products.find({
    'categories': 'oats'
}).explain()
exp['executionStats']

{'executionSuccess': True,
 'nReturned': 21,
 'executionTimeMillis': 0,
 'totalKeysExamined': 0,
 'totalDocsExamined': 200,
 'executionStages': {'isCached': False,
  'stage': 'COLLSCAN',
  'filter': {'categories': {'$eq': 'oats'}},
  'nReturned': 21,
  'executionTimeMillisEstimate': 0,
  'works': 201,
  'advanced': 21,
  'needTime': 179,
  'needYield': 0,
  'saveState': 0,
  'restoreState': 0,
  'isEOF': 1,
  'direction': 'forward',
  'docsExamined': 200},
 'allPlansExecution': []}

In [9]:
products.create_index({
    'categories': 1,
    'specs.moisture': 1
})

products.create_index({
    'description': 'text'
})

'description_text'

In [10]:
# [item for item in products.find()]
exp = products.find({
    'categories': 'oats'
}).explain()
exp['executionStats']

{'executionSuccess': True,
 'nReturned': 21,
 'executionTimeMillis': 1,
 'totalKeysExamined': 21,
 'totalDocsExamined': 21,
 'executionStages': {'isCached': False,
  'stage': 'FETCH',
  'nReturned': 21,
  'executionTimeMillisEstimate': 0,
  'works': 22,
  'advanced': 21,
  'needTime': 0,
  'needYield': 0,
  'saveState': 0,
  'restoreState': 0,
  'isEOF': 1,
  'docsExamined': 21,
  'alreadyHasObj': 0,
  'inputStage': {'stage': 'IXSCAN',
   'nReturned': 21,
   'executionTimeMillisEstimate': 0,
   'works': 22,
   'advanced': 21,
   'needTime': 0,
   'needYield': 0,
   'saveState': 0,
   'restoreState': 0,
   'isEOF': 1,
   'keyPattern': {'categories': 1, 'specs.moisture': 1},
   'indexName': 'categories_1_specs.moisture_1',
   'isMultiKey': False,
   'multiKeyPaths': {'categories': [], 'specs.moisture': []},
   'isUnique': False,
   'isSparse': False,
   'isPartial': False,
   'indexVersion': 2,
   'direction': 'forward',
   'indexBounds': {'categories': ['["oats", "oats"]'],
    'specs.m

### Aggregation Pipeline

In [19]:
pipeline = [{
"$match": {
    "created_at": {
    "$gte": datetime.now(UTC) - timedelta(days=30)}
}
},

{
"$lookup": {
    "from": "products",
    "localField": "product_id",
    "foreignField": "_id",
    "as": "product"}
},

{
"$unwind": "$product" # :[{}] -> :{}
},

{
"$group": {
    "_id": "$product.categories",
    "avg_price": {"$avg": "$price"},
    "lots": {"$sum": 1}}
},
{
"$project": {
    "category": "$_id",
    "price_avg": {"$round": ["$avg_price", 2]},
    "lots": 1,
    "_id": 0
}
}
]

[i for i in orders.aggregate(pipeline)]

[{'lots': 7, 'category': 'soy', 'price_avg': 360.29},
 {'lots': 12, 'category': 'oats', 'price_avg': 261.08},
 {'lots': 9, 'category': 'corn', 'price_avg': 419.89},
 {'lots': 15, 'category': 'millet', 'price_avg': 408.67},
 {'lots': 16, 'category': 'chickpeas', 'price_avg': 357.5},
 {'lots': 11, 'category': 'wheat', 'price_avg': 350.27},
 {'lots': 11, 'category': 'beans', 'price_avg': 350.27},
 {'lots': 14, 'category': 'rapeseed', 'price_avg': 328.79},
 {'lots': 7, 'category': 'barley', 'price_avg': 344.57}]

In [13]:
pipeline = [

{
"$group": {
    "_id": "$seller_id",
    "value": {"$sum": {"$multiply": ["$quantity","$price"]}}
}
},
{"$sort": {"value": -1}},
{"$limit": 10}
]

[i for i in orders.aggregate(pipeline)]

[{'_id': ObjectId('69aedd229c2f5d0156cb1b2f'), 'value': 2731399},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b64'), 'value': 2582429},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b9d'), 'value': 2401728},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b4a'), 'value': 2281614},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b25'), 'value': 2170880},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b79'), 'value': 2152919},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bab'), 'value': 2107218},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b06'), 'value': 1932506},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b6c'), 'value': 1917704},
 {'_id': ObjectId('69aedd229c2f5d0156cb1b03'), 'value': 1906627}]

In [43]:
pipeline = [
{
"$lookup": {
    "from": "products",
    "localField": "product_id",
    "foreignField": "_id",
    "as": "product"}
},
{"$unwind": "$product"},
{"$match": {"product.categories": "wheat"}},
{
"$group": {
    "_id": {"$week": "$created_at"},
    "avg": {"$avg": "$price"}
}
},
{"$sort": {"_id": 1}},
{
"$project": {
    "week": "$_id",
    "avg": {"$round": ["$avg", 2]},
    "_id": 0
}}
]

list(orders.aggregate(pipeline))

[{'week': 0, 'avg': 548.0},
 {'week': 1, 'avg': 346.5},
 {'week': 2, 'avg': 400.0},
 {'week': 3, 'avg': 403.0},
 {'week': 5, 'avg': 542.0},
 {'week': 6, 'avg': 532.5},
 {'week': 7, 'avg': 257.6},
 {'week': 8, 'avg': 353.33},
 {'week': 9, 'avg': 440.0},
 {'week': 10, 'avg': 521.0},
 {'week': 11, 'avg': 395.0},
 {'week': 12, 'avg': 625.0},
 {'week': 13, 'avg': 226.0},
 {'week': 14, 'avg': 459.67},
 {'week': 16, 'avg': 401.0},
 {'week': 17, 'avg': 460.75},
 {'week': 18, 'avg': 337.6},
 {'week': 19, 'avg': 127.0},
 {'week': 20, 'avg': 404.0},
 {'week': 21, 'avg': 407.25},
 {'week': 22, 'avg': 374.5},
 {'week': 23, 'avg': 574.0},
 {'week': 25, 'avg': 509.43},
 {'week': 26, 'avg': 384.33},
 {'week': 27, 'avg': 468.5},
 {'week': 28, 'avg': 385.67},
 {'week': 30, 'avg': 483.0},
 {'week': 31, 'avg': 481.0},
 {'week': 32, 'avg': 346.0},
 {'week': 33, 'avg': 280.57},
 {'week': 35, 'avg': 365.6},
 {'week': 36, 'avg': 424.5},
 {'week': 37, 'avg': 397.0},
 {'week': 38, 'avg': 436.0},
 {'week': 39, '

In [46]:
pipeline = [
{
"$lookup": {
    "from": "sellers",
    "localField": "seller_id",
    "foreignField": "_id",
    "as": "seller"}
},
{"$unwind": "$seller"},
{
"$project": {
    "quantity": 1,
    "price": 1,
    "seller_name": "$seller.name",
    "country": "$seller.country"
}}
]

list(orders.aggregate(pipeline))

[{'_id': ObjectId('69aedd229c2f5d0156cb1bb2'),
  'quantity': 115,
  'price': 275,
  'seller_name': 'Smith, Wong and Higgins',
  'country': 'Ireland'},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bb3'),
  'quantity': 195,
  'price': 325,
  'seller_name': 'Foley-Jones',
  'country': 'Lesotho'},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bb4'),
  'quantity': 290,
  'price': 380,
  'seller_name': 'Anderson, Shaw and Rivera',
  'country': 'Portugal'},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bb5'),
  'quantity': 257,
  'price': 579,
  'seller_name': 'Lawson-Thomas',
  'country': 'British Indian Ocean Territory (Chagos Archipelago)'},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bb6'),
  'quantity': 216,
  'price': 513,
  'seller_name': 'Mcbride PLC',
  'country': 'Martinique'},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bb7'),
  'quantity': 792,
  'price': 491,
  'seller_name': 'Johnson Ltd',
  'country': 'Latvia'},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bb8'),
  'quantity': 387,
  'price': 456,
  'seller

### ReplicaSet

### Postgres data

In [5]:
sys.path.append(os.path.abspath(".."))

In [6]:
from sqlalchemy.orm import Session
from models.models import Base, Teacher, Course, Lesson, Student, Enrollment, Submission
from postgres_db import SessionLocal
from faker import Faker
import random
from datetime import datetime

fake = Faker()
session = SessionLocal()

In [7]:
teachers = []
for _ in range(5):  # 5 преподавателей
    t = Teacher(name=fake.name())
    teachers.append(t)
session.add_all(teachers)
session.commit()

In [8]:
courses = []
for teacher in teachers:
    for _ in range(10):
        c = Course(title=fake.sentence(nb_words=3), teacher=teacher)
        courses.append(c)
session.add_all(courses)
session.commit()

In [9]:
lessons = []
for course in courses:
    for order_num in range(1, random.randint(3, 8)):
        l = Lesson(title=f"Lesson {order_num} of {course.title}", course=course, order_num=order_num)
        lessons.append(l)
session.add_all(lessons)
session.commit()

In [10]:
students = []
for _ in range(50):
    s = Student(name=fake.name())
    students.append(s)
session.add_all(students)
session.commit()

In [11]:
enrollments = []
for student in students:
    chosen_courses = random.sample(courses, k=random.randint(3, 6))
    for course in chosen_courses:
        e = Enrollment(
            student=student,
            course=course,
            enrolled_at=fake.date_time_this_year(),
            progress_pct=random.randint(0, 100),
            certificate_url=None
        )
        enrollments.append(e)
session.add_all(enrollments)
session.commit()

In [12]:
submissions = []
for enrollment in enrollments:
    course = enrollment.course
    for lesson in course.lessons:
        s = Submission(
            student=enrollment.student,
            lesson=lesson,
            score=random.randint(0, 100)
        )
        submissions.append(s)
session.add_all(submissions)
session.commit()

session.close()

### Functions

In [13]:
sys.path.append(os.path.abspath(".."))

In [14]:
import time
from sqlalchemy import func
from sqlalchemy.orm import selectinload, joinedload, contains_eager
from postgres_db import SessionLocal
from models.models import Course, Enrollment, Submission, Teacher, Lesson, Student 

In [15]:
def get_course_stats(course_id: int):
    session = SessionLocal()

    students_count = session.query(func.count(Enrollment.student_id))\
        .filter(Enrollment.course_id == course_id).scalar()

    avg_progress = session.query(func.avg(Enrollment.progress_pct))\
        .filter(Enrollment.course_id == course_id).scalar() or 0
    
    completed_count = session.query(func.count(Enrollment.student_id))\
        .filter(Enrollment.course_id == course_id, Enrollment.progress_pct == 100).scalar()
    completed_pct = (completed_count / students_count * 100) if students_count else 0
    
    total_revenue = session.query(func.sum(Submission.score))\
        .join(Lesson, Lesson.id == Submission.lesson_id)\
        .filter(Lesson.course_id == course_id).scalar() or 0
    
    session.close()
    return {
        "students_count": students_count,
        "avg_progress": avg_progress,
        "completed_pct": completed_pct,
        "total_revenue": total_revenue
    }

In [16]:
def top_teachers(n: int = 5):
    session = SessionLocal()
    subquery = session.query(
        Lesson.course_id.label("course_id"),
        func.avg(Submission.score).label("avg_score")
    ).join(Submission, Submission.lesson_id == Lesson.id)\
     .group_by(Lesson.course_id).subquery()
    result = session.query(
        Teacher,
        func.avg(subquery.c.avg_score).label("teacher_rating")
    ).join(Course, Course.teacher_id == Teacher.id)\
     .join(subquery, subquery.c.course_id == Course.id)\
     .group_by(Teacher.id)\
     .order_by(func.avg(subquery.c.avg_score).desc())\
     .limit(n).all()
    
    session.close()
    return [(t.name, float(r)) for t,r in result]

In [17]:
def load_courses_lazy():
    session = SessionLocal()
    start = time.time()
    courses = session.query(Course).all()
    for c in courses:
        _ = c.lessons
    elapsed = time.time() - start
    session.close()
    return elapsed

def load_courses_eager():
    session = SessionLocal()
    start = time.time()
    courses = session.query(Course).options(joinedload(Course.lessons)).all()
    elapsed = time.time() - start
    session.close()
    return elapsed

In [18]:
def n_plus_1_example(course_id: int):
    session = SessionLocal()
    course = session.query(Course).filter(Course.id == course_id).first()
    results = []
    for enrollment in course.students_assoc:
        student = enrollment.student
        subs = session.query(Submission).join(Lesson).filter(
            Lesson.course_id==course_id,
            Submission.student_id==student.id
        ).all()
        results.append((student.name, [s.score for s in subs]))
    session.close()
    return results

def n_plus_1_fixed(course_id: int):
    session = SessionLocal()
    course = session.query(Course)\
        .options(
            selectinload(Course.students_assoc)
            .selectinload(Enrollment.student)
        ).filter(Course.id==course_id).first()
    lesson_scores = session.query(Submission)\
        .join(Lesson)\
        .filter(Lesson.course_id==course_id).all()
    student_map = {e.student.id: e.student.name for e in course.students_assoc}
    results = {name: [] for name in student_map.values()}
    for s in lesson_scores:
        if s.student_id in student_map:
            results[student_map[s.student_id]].append(s.score)
    
    session.close()
    return results

### Functions using

In [ ]:
print(get_course_stats(1))

In [26]:
print(top_teachers())

[('Leonard Patrick', 53.51331349206348), ('Michael Morris', 51.54769841269841), ('Matthew Miller', 50.65238095238095), ('Victoria Hernandez', 46.268251700680274), ('Robert Curry', 45.8483798185941)]


In [30]:
print(f'{load_courses_lazy():.6f}')
print(f'{load_courses_eager():.6f}')

0.068520
0.006418


In [24]:
results = n_plus_1_example(1)
for student, scores in results:
    print(student, scores)

Nathaniel Ho [78.0, 14.0, 48.0, 3.0, 84.0]
John Wright [13.0, 55.0, 56.0, 36.0, 99.0]
Tammy Walker [3.0, 70.0, 13.0, 7.0, 87.0]


In [25]:
results = n_plus_1_fixed(1)
for student, scores in results.items():
    print(student, scores)

Nathaniel Ho [78.0, 14.0, 48.0, 3.0, 84.0]
John Wright [13.0, 55.0, 56.0, 36.0, 99.0]
Tammy Walker [3.0, 70.0, 13.0, 7.0, 87.0]


### RAW & ORM

In [35]:
from sqlalchemy import case, func

def course_analytics_orm():
    session = SessionLocal()
    start = time.time()
    
    results = session.query(
        Course.id,
        Course.title,
        func.count(Enrollment.student_id).label("students_count"),
        func.avg(Enrollment.progress_pct).label("avg_progress"),
        func.sum(case((Enrollment.progress_pct > 50, 1), else_=0)).label("above_50")
    ).join(Enrollment, Enrollment.course_id == Course.id)\
     .group_by(Course.id).all()
    
    elapsed = time.time() - start
    session.close()
    return results, elapsed

In [36]:
from sqlalchemy import text

def course_analytics_raw():
    session = SessionLocal()
    start = time.time()
    
    sql = text("""
        SELECT 
            c.id,
            c.title,
            COUNT(e.student_id) AS students_count,
            AVG(e.progress_pct) AS avg_progress,
            SUM(CASE WHEN e.progress_pct > 50 THEN 1 ELSE 0 END) AS above_50
        FROM course c
        JOIN enrollment e ON e.course_id = c.id
        GROUP BY c.id, c.title
    """)
    
    result = session.execute(sql).all()
    elapsed = time.time() - start
    session.close()
    return result, elapsed

In [42]:
orm_res, orm_time = course_analytics_orm()
raw_res, raw_time = course_analytics_raw()

print("ORM:", orm_res, f'\n\n({orm_time:.6f}s)')
print(f'\n\n')
print("Raw SQL:", raw_res, f'\n\n({raw_time:.6f})')

ORM: [(22, 'Marriage appear.', 3, 50.0, 1), (11, 'Camera people.', 4, 50.0, 2), (44, 'Approach create.', 3, 80.0, 2), (42, 'Direction necessary.', 2, 88.0, 2), (40, 'Mother fire.', 3, 28.333333333333332, 0), (43, 'List standard.', 2, 52.5, 1), (9, 'Air environment.', 3, 39.333333333333336, 1), (15, 'Different mention third.', 8, 35.5, 3), (26, 'Ball finish remain.', 5, 32.8, 1), (48, 'Process quality significant go.', 6, 31.333333333333332, 1), (19, 'Condition house then.', 3, 60.333333333333336, 2), (30, 'Gas away room early.', 6, 56.333333333333336, 3), (21, 'Sometimes do modern.', 2, 29.0, 0), (3, 'Beautiful prepare six.', 4, 52.0, 2), (17, 'I reveal community.', 7, 45.714285714285715, 4), (28, 'Picture present.', 2, 40.5, 1), (37, 'Pick discussion.', 7, 61.857142857142854, 4), (5, 'Could hand perform.', 6, 41.333333333333336, 1), (29, 'Student whole mother.', 7, 65.57142857142857, 5), (4, 'Race reveal if.', 7, 59.0, 5), (34, 'Hold each.', 4, 68.25, 3), (10, 'Prepare strong.', 3, 43

### Tasks

In [23]:
orders.find_one()


{'_id': ObjectId('69aedd229c2f5d0156cb1bb2'),
 'product_id': ObjectId('69aedd229c2f5d0156cb1abb'),
 'seller_id': ObjectId('69aedd229c2f5d0156cb1b03'),
 'quantity': 115,
 'price': 275,
 'created_at': datetime.datetime(2025, 9, 17, 22, 20, 54, 863000)}

In [27]:
pipeline = [
{
"$lookup": {
    "from": "products",
    "localField": "product_id",
    "foreignField": "_id",
    "as": "product"
}
},
{"$unwind": "$product"}
]

list(orders.aggregate(pipeline))[:3]

[{'_id': ObjectId('69aedd229c2f5d0156cb1bb2'),
  'product_id': ObjectId('69aedd229c2f5d0156cb1abb'),
  'seller_id': ObjectId('69aedd229c2f5d0156cb1b03'),
  'quantity': 115,
  'price': 275,
  'created_at': datetime.datetime(2025, 9, 17, 22, 20, 54, 863000),
  'product': {'_id': ObjectId('69aedd229c2f5d0156cb1abb'),
   'name': 'financial',
   'categories': 'soy',
   'description': 'Area break particularly thing population or. Behind rise inside culture according project nice.\nAhead plan daughter magazine. Out identify group laugh resource. Quality box sense director notice.',
   'specs': {'moisture': 10.23, 'protein': 49.49, 'gluten': 13.77}}},
 {'_id': ObjectId('69aedd229c2f5d0156cb1bb3'),
  'product_id': ObjectId('69aedd229c2f5d0156cb1abd'),
  'seller_id': ObjectId('69aedd229c2f5d0156cb1b0e'),
  'quantity': 195,
  'price': 325,
  'created_at': datetime.datetime(2026, 1, 12, 11, 56, 54, 864000),
  'product': {'_id': ObjectId('69aedd229c2f5d0156cb1abd'),
   'name': 'few',
   'categories

In [11]:
item_1 = {
  "_id" : "U1IT00001",
  "item_name" : "Blender",
  "max_discount" : "10%",
  "batch_number" : "RR450020FRG",
  "price" : 340,
  "category" : "kitchen appliance"
}

item_2 = {
  "_id" : "U1IT00002",
  "item_name" : "Egg",
  "category" : "food",
  "quantity" : 12,
  "price" : 36,
  "item_description" : "brown country eggs"
}
products.insert_many([item_1,item_2])

BulkWriteError: batch op errors occurred, full error: {'writeErrors': [{'index': 0, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: agro_test.products index: _id_ dup key: { _id: "U1IT00001" }', 'keyPattern': {'_id': 1}, 'keyValue': {'_id': 'U1IT00001'}, 'op': {'_id': 'U1IT00001', 'item_name': 'Blender', 'max_discount': '10%', 'batch_number': 'RR450020FRG', 'price': 340, 'category': 'kitchen appliance'}}], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 0, 'nMatched': 0, 'nModified': 0, 'nRemoved': 0, 'upserted': []}

In [18]:
products.find({'item_name': 'Egg'})

In [42]:
[item for item in products.find()]

[{'_id': ObjectId('69aea46b5d643f0a54fe50f2'),
  'name': 'low',
  'categories': 'soy',
  'description': 'Way necessary hospital rather assume through. Leader must assume black candidate. Game whether culture all.',
  'specs': {'moisture': 12.66, 'protein': 41.7, 'gluten': 26.29}},
 {'_id': ObjectId('69aea46b5d643f0a54fe50f3'),
  'name': 'life',
  'categories': 'oats',
  'description': 'Research magazine figure large. Foot hold wife later shoulder general not phone.\nFear check picture local.\nPlay sort seat rich. Get direction doctor mouth executive subject effect.',
  'specs': {'moisture': 2.73, 'protein': 27.42, 'gluten': 2.37}},
 {'_id': ObjectId('69aea46b5d643f0a54fe50f4'),
  'name': 'rock',
  'categories': 'beans',
  'description': 'Approach season tell. Difficult catch add major. Maintain task face position challenge list success usually. Line might talk.',
  'specs': {'moisture': 0.5, 'protein': 38.34, 'gluten': 29.21}},
 {'_id': ObjectId('69aea46b5d643f0a54fe50f5'),
  'name': '

In [ ]:
📝 Задание 3 — MongoDB: система аналитики для агро-маркетплейса
Инструменты: MongoDB 7+, PyMongo / Motor (async), Python
Условие
Смоделировать и реализовать хранилище данных маркетплейса сельскохозяйственной
продукции с аналитическими возможностями.
Контекст: каждый лот на маркетплейсе — это продукт с переменным набором атрибутов
(зерно имеет влажность и клейковину, молоко — жирность и белок и т.д.). Реляционная схема
здесь неудобна — отличный кейс для документной БД.
Задачи:
    1. Проектирование схемы документов — спроектировать коллекции products, sellers, orders, price_history.
        В products атрибуты хранить как вложенный объект specs: { moisture: 14.2, protein: 12.5, ... }.
        Обосновать выбор embedding vs referencing для orders → products.
    2. CRUD + индексы:
        ◦ Вставить 1000+ документов (генерация через faker).
        ◦ Создать составной индекс { category: 1, "specs.moisture": 1 } и text-индекс на поле description.
        ◦ Сравнить explain("executionStats") до и после индексирования.
    3. Aggregation Pipeline — реализовать следующие запросы:
        ◦ Средняя цена и количество лотов по категориям за последние 30 дней.
        ◦ Топ-10 продавцов по объёму продаж (сумма qty × price).
        ◦ Динамика цен на пшеницу по неделям (группировка по $week).
        ◦ $lookup — объединить заказы с данными продавца (аналог JOIN).
    4. Change Streams (опционально, +бонус) — подписаться на изменения в коллекции price_history и логировать каждое изменение цены более чем на 10%.
    5. Сравнение с SQL — в файле README.md (или ячейке notebook) ответить на вопросы:
        ◦ В каком сценарии MongoDB выигрывает у PostgreSQL для данного домена?
        ◦ Почему Redis не подходит в качестве основного хранилища здесь?
        ◦ Как бы выглядела нормализованная SQL-схема и какие JOIN'ы потребовались бы?

In [7]:
for _ in range(10):
    print(fake.company())


Farmer, Jones and Fisher
Thompson-Meyer
Mayer, Villarreal and Mcconnell
Sanchez PLC
Boone-Riley
Golden, Weber and Williams
Rodriguez Ltd
Bond-Contreras
Allen-Brown
Jones, Hale and Higgins
